## Field-Noyes Oregonator-style Model System

$$\begin{align*}
A + Y \to X + P \\ 
X + Y \to 2P \\ 
A + X \to 2X + 2Z \\ 
2X \to A+P \\ 
B + Z \to (f/2)Y
\end{align*}$$

from https://www.ebi.ac.uk/biomodels/BIOMD0000000040?

k1 = 1.34, k2 = 1.6×10^9, k3 = 8000, k4 = 4.0×10^7


In [12]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# pyrdme imports
from pyrdme import (
    Lattice2D,
    Reaction,
    simulate_rdme,
    simulate_diffusion_only,
    validate_timestep,
)

# Set random seed for reproducibility
np.random.seed(42)

In [13]:
# Create a 100x100 lattice with 1 μm spacing
lattice = Lattice2D(
    nx=300,
    ny=300,
    species=['A', 'B', 'X', 'Y', 'Z', 'P', 'Q'],
    spacing=1e-6,  # 1 μm
)

print(lattice)



Lattice2D(300×300, 7 species, 0 particles)


In [14]:
# Add reactants uniformly across the lattice
lattice.add_particles('A', count=1000, distribution='random')
lattice.add_particles('B', count=1000, distribution='random')

# Seed intermediates
lattice.add_particles('X', count=5, distribution='random')
lattice.add_particles('Y', count=5, distribution='random')
lattice.add_particles('Z', count=5, distribution='random')

# Add products
lattice.add_particles('P', count=5, distribution='random')
lattice.add_particles('Q', count=5, distribution='random')



In [ ]:
Ks = [1, 1, 1, 1, 1]
reactions = [
    Reaction(['A', 'Y'], ['X'], k=Ks[0]),   
    Reaction(['X', 'Y'], ['P'], k=Ks[1]),   
    Reaction(['B', 'X'], ['X', 'X', 'Z'], k=Ks[2]),
    Reaction(['X', 'X'], ['Q'], k=Ks[3]),
    Reaction(['Z'], ['Y'], k=Ks[4]),
]

diffusion = {
    'A': 2e-9,
    'B': 1e-9,
    'X': 0.5e-10, 
    'Y': 1e-10,
    'Z': 1e-10,
    'P': 0.33e-10,
    'Q': 0.25e-10,
}

In [16]:
# Run simulation
result = simulate_rdme(
    lattice=lattice,
    reactions=reactions,
    diffusion=diffusion,
    t_max=0.5,  # 500 ms
    seed=42,
)

In [21]:
# Create animation of species P formation
fig, ax = plt.subplots(figsize=(8, 6))

# Get all count arrays for C
counts_P_all = result.get_counts('X')  # Shape: (n_times, nx, ny)
vmax = counts_P_all.max()

im = ax.imshow(counts_P_all[0].T, origin='lower', cmap='viridis', vmin=0, vmax=vmax)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
title = ax.set_title('Species P, t = 0 ms', fontsize=14)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Count', fontsize=11)

def animate(frame):
    im.set_array(counts_P_all[frame].T)
    title.set_text(f'Species P, t = {result.times[frame]*1000:.0f} ms')
    return [im, title]

anim = FuncAnimation(fig, animate, frames=len(result.times), interval=200, blit=True)
plt.close()  # Prevent static display

# Display animation
HTML(anim.to_jshtml())

In [17]:
# Create animations for all species with distinct colormaps
species_cmaps = {
    'A': 'Reds',
    'B': 'Blues',
    'X': 'Greens',
    'Y': 'Purples',
    'Z': 'Oranges',
    'P': 'viridis',
    'Q': 'plasma',
}

for species, cmap in species_cmaps.items():
    fig, ax = plt.subplots(figsize=(8, 6))

    counts_all = result.get_counts(species)  # Shape: (n_times, nx, ny)
    vmax = max(counts_all.max(), 1)  # Avoid vmax=0

    im = ax.imshow(counts_all[0].T, origin='lower', cmap=cmap, vmin=0, vmax=vmax)
    ax.set_xlabel('x', fontsize=12)
    ax.set_ylabel('y', fontsize=12)
    title = ax.set_title(f'Species {species}, t = 0 ms', fontsize=14)
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Count', fontsize=11)

    def make_animate(counts, sp, im_ref, title_ref):
        def animate(frame):
            im_ref.set_array(counts[frame].T)
            title_ref.set_text(f'Species {sp}, t = {result.times[frame]*1000:.0f} ms')
            return [im_ref, title_ref]
        return animate

    anim = FuncAnimation(
        fig,
        make_animate(counts_all, species, im, title),
        frames=len(result.times),
        interval=200,
        blit=True,
    )
    plt.close()
    display(HTML(anim.to_jshtml()))

KeyboardInterrupt: 